In [1]:
# For now, we still need to manually add atomworks dependency
import sys;
sys.path.insert(0, '/home/jbutch/Projects/HT25/af3/rfd3-release/lib/atomworks/src')
sys.path.append('/home/jbutch/Projects/HT25/af3/rfd3-release/models/rfd3/src')
sys.path.append('/home/jbutch/Projects/HT25/af3/rfd3-release/src')

from atomworks.io.utils.visualize import view

Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


# 1) Design with RFD3

In [ ]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from rfd3.inference.input_parsing import DesignInputSpecification
conf = RFD3InferenceConfig(
    ckpt_path='/projects/ml/aa_design/models/rfd3_latest_cleaned.ckpt',
    diffusion_batch_size=2,
    dump_trajectories=True,
    inference_sampler={"num_timesteps": 10},
)
# Lazily initialize model
model = RFD3InferenceEngine(**conf)

18:33:18 INFO modelhub.inference_engines.base: [rank: 0] Seed is None - using external RNG state


In [3]:
spec = DesignInputSpecification(
    input='/home/jbutch/Projects/HT25/af3/rfd3-release/models/rfd3/tests/test_data/amidase_LOW.pdb',
    length=10,
    ligand='L:G',
    unindex='A92-96,B2,E5',
    select_fixed_atoms={
        'B2': 'ND1,CG,NE2,CD2,CB,CE1',
        'E5': "OH,CZ,CE1,CE2,CD2,CD1,CG,CB"
    }
)
view(spec.build())

18:33:18 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.
18:33:18 INFO rfd3.utils.inference: [rank: 0] No ori_token or infer_ori_strategy provided. Setting origin as COM of fixed motif ([ 0.08121534 -0.06559487 -0.11668349]).


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
outputs = model.run(
    inputs=spec,
    n_batches=1,
)

18:33:18 INFO modelhub.inference_engines.base: [rank: 0] Loading checkpoint from /projects/ml/aa_design/models/rfd3_latest_cleaned.ckpt...
18:33:20 ERROR modelhub.utils.ddp: No GPUs available - Setting accelerator to 'cpu'. Are you sure you are using the correct configs?
18:33:20 INFO modelhub.inference_engines.base: [rank: 0] Building Transform pipeline...
18:33:20 INFO modelhub.inference_engines.base: [rank: 0] Using settings from validation dataset: unconditional.
18:33:20 INFO rdkit: Enabling RDKit 2025.03.6 jupyter extensions
18:33:21 INFO modelhub.inference_engines.base: [rank: 0] Instantiating trainer...
/home/jbutch/Projects/HT25/af3/rfd3-release/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jbutch/Projects/HT25/af3/rfd3-release/.venv/li ...
Using bfloat16

In [7]:
for idx, data in outputs.items():
    print(f"Output type for batch {idx}: {type(data)}[0] = {type(data[0])}")
    print(f"Output atom_array: {data[0].atom_array}")
    atom_array = data[0].atom_array
view(atom_array)

Output type for batch backbone_0_0: <class 'list'>[0] = <class 'rfd3.engine.RFD3Output'>
Output atom_array:     A       1  HIS N      N         6.247   -1.579    4.615
    A       1  HIS CA     C         5.734   -2.402    3.545
    A       1  HIS C      C         5.578   -3.849    3.931
    A       1  HIS O      O         6.383   -4.703    3.523
    A       1  HIS CB     C         3.234    3.234    5.500
    A       1  HIS CG     C         3.062    3.109    4.031
    A       1  HIS ND1    N         3.922    3.719    3.125
    A       1  HIS CD2    C         2.156    2.453    3.266
    A       1  HIS CE1    C         3.516    3.422    1.875
    A       1  HIS NE2    N         2.438    2.641    1.922
    A       2  TYR N      N         4.495   -4.204    4.611
    A       2  TYR CA     C         4.326   -5.563    5.109
    A       2  TYR C      C         2.897   -6.048    5.031
    A       2  TYR O      O         2.663   -7.238    5.234
    A       2  TYR CB     C         7.312   -2.484  

3Dmol.js failed to load for some reason. Please check your browser console for error messages.